# Company Brochure Generator
### This program generates a professional company brochure by providing a company name and URL.
### It uses the Anthropic Claude API with built-in web search to research the company
### and stream a formatted Markdown brochure in real time inside a Jupyter notebook.

In [2]:
import os                          # For accessing environment variables
import anthropic                   # Anthropic Python SDK for Claude API access
from dotenv import load_dotenv     # Loads environment variables from a .env file
from IPython.display import display, Markdown, update_display  # Jupyter display utilities for live Markdown rendering
import gradio as gr

In [3]:
## Load Environment Variables
load_dotenv(verbose=True)

True

In [4]:
# Validate the Anthropic API Key
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if anthropic_api_key and anthropic_api_key.startswith("sk-ant-") and len(anthropic_api_key) >10:
    print(f"anthropic api key is Valid")
else:
    print("Anthropic API key not found or invalid — check your .env file")

anthropic api key is Valid


In [5]:
# Set the Claude model to use. `claude-sonnet-4-6` balances speed, cost, and quality.
MODEL = "claude-sonnet-4-6"

In [6]:
# Initialise the Anthropic Client
# Creates a reusable client instance authenticated with the validated API key
anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)

In [7]:
SYSTEM_PROMPT = '''
You are an expert business analyst and copywriter.
When given a company name and URL, you will:
1. Use web search to explore the company's website, about page, careers, culture, and any other relevant pages.
2. Synthesize everything into a short, compelling company brochure in Markdown.
3. The brochure should cover: what the company does, its culture/values, who its customers are, and career opportunities if available.
4. Write in clean Markdown without code blocks.
'''

In [10]:
def stream_brochure_generate(company_name: str, url: str) -> None:
    """
    Streaming version.
    Renders the brochure live in markdown as Claude writes it.
    """
    print(f"Researching {company_name}... Claude will search the web automatically.\n")
    # Open a streaming context — Claude will emit tokens as it generates them
    with anthropic_client.messages.stream(
        model=MODEL,
        max_tokens=2000,
        system=SYSTEM_PROMPT,
        # Grant Claude access to web search (limited to 5 calls to control latency/cost)
        tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}],
        messages=[{"role": "user", "content": f"Research {company_name} at {url} and create a professional brochure."}],
    ) as stream:
        full_response = ""
        # Create an initial empty Markdown display cell; we'll update it as tokens arrive.
        # display_handler = display(Markdown(""), display_id=True)
        # Iterate over each text chunk emitted by the stream
        for text in stream.text_stream:
            full_response += text
            # Re-render the entire Markdown so the cell updates live
            #update_display(Markdown(full_response), display_id=display_handler.display_id)
            yield full_response

In [14]:
name_input = gr.Textbox(label="Company Name")
url_input = gr.Textbox(label="Company URL including http:// or https://")
message_output = gr.Markdown(label="Response : ")

view = gr.Interface(
    fn=stream_brochure_generate,
    inputs=[name_input, url_input],
    outputs=[message_output],
    examples=[
        ["hugging face", "https://huggingface.co"],
        ["follett", "https://follett.com"]
    ],
    flagging_mode="never"
)

view.launch(share=True)


* Running on local URL:  http://127.0.0.1:7869
* Running on public URL: https://667e03604b045fdaba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


'\nif __name__ == "__main__":\n    user_input = input("Enter company name and URL separated by a comma\n  example: Hugging Face, https://huggingface.co : ")\n    print(f"\n{user_input}\n")\n\n    # Split the input into two parts on the first comma only\n    parts = user_input.split(",", 1)\n\n    if len(parts) != 2:\n        print("Invalid input. Please enter both a company name and a URL separated by a comma.")\n    else:\n        company_name = parts[0].strip()  # Remove accidental leading/trailing spaces\n        url = parts[1].strip()\n        stream_brochure_generate(company_name, url)\n'

Researching follett... Claude will search the web automatically.

Researching follett... Claude will search the web automatically.

